In [0]:
dbutils.help()

In [0]:
dbutils.secrets.help()


In [0]:
dbutils.secrets.listScopes()

In [0]:
dbutils.secrets.list(scope="practicescope")

In [0]:
dbutils.secrets.get(scope="practicescope", key="clientid")
dbutils.secrets.get(scope="practicescope", key="password")


In [0]:
    spark.conf.set(
        "fs.azure.account.auth.type.newpracticesa.dfs.core.windows.net",
        "OAuth"
    )

    spark.conf.set(
        "fs.azure.account.oauth.provider.type.newpracticesa.dfs.core.windows.net",
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
    )

    spark.conf.set(
        "fs.azure.account.oauth2.client.id.newpracticesa.dfs.core.windows.net",
        dbutils.secrets.get(scope="practicescope", key="clientid")
    )

    spark.conf.set(
        "fs.azure.account.oauth2.client.secret.newpracticesa.dfs.core.windows.net",
        dbutils.secrets.get(scope="practicescope", key="password")
    )

    spark.conf.set(
        "fs.azure.account.oauth2.client.endpoint.newpracticesa.dfs.core.windows.net",
        "https://login.microsoftonline.com/bda11e34-4864-4eb6-82cb-acbfb3913d55/oauth2/v2.0/token"
    )


In [0]:
df = spark.read.format("csv").option("header","true").option("inferSchema","true").load(
    "abfss://bronze@newpracticesa.dfs.core.windows.net/"
)
display(df)

In [0]:
df.count()

In [0]:
df.dropDuplicates().count()

In [0]:
df.select("region").distinct().show()

In [0]:
from pyspark.sql.functions import regexp_replace
df = df.withColumn("region", regexp_replace("region", "souht", "south"))


In [0]:
df.select("region").distinct().show()

In [0]:
from pyspark.sql.functions import col, when
df = df.withColumn("region", 
                   when(col("region") == "south", "South")
                   .when(col("region")=="Wset","West")
                   .otherwise(col("region")))
df.select("region").distinct().show()


In [0]:
df.select("gender").distinct().show()

In [0]:
df = df.replace({'Mal': 'Male', 'males': 'Male', 'females': 'Female'}, subset=['gender'])


In [0]:
silver_path = "abfss://silver@newpracticesa.dfs.core.windows.net/customer"

df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
silver_path = "abfss://silver@newpracticesa.dfs.core.windows.net/customer/"

dbutils.fs.mkdirs(silver_path)
dbutils.fs.ls(silver_path)

In [0]:
    df.write \
    .format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(silver_path)

In [0]:
df_silver = spark.read.format("delta").load(silver_path)

display(df_silver)

In [0]:
csv_path = "abfss://silver@newpracticesa.dfs.core.windows.net/customermaster_csv/"
df.coalesce(1) \
    .write \
    .format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(csv_path)    

In [0]:
csv_path = "abfss://silver@newpracticesa.dfs.core.windows.net/customermaster_csv/"
df_csv = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(csv_path)
display(df_csv)